# Clasificador OpenAI gpt-4o-mini
Clasifica el dataset de preguntas de red usando la API de OpenAI.

In [ ]:
%pip install openai pandas tqdm

## Configuración

In [ ]:
API_KEY    = "<tu_api_key>"
MODEL      = "gpt-4o-mini"
INPUT_CSV  = "requirements_questions_v2.csv"
OUTPUT_CSV = "requirements_classified_openai.csv"
CHECKPOINT = "checkpoint_openai.json"
SAMPLE     = 20   # None para el dataset completo

## Setup

In [ ]:
import re
import json
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

CAT_LIST = ["ROUTING", "SECURITY", "QOS", "CONNECTIVITY", "MONITORING", "GENERAL"]

SYSTEM_PROMPT = """You are a Cisco network expert and dataset annotator.
Classify the given network configuration question into exactly one of these categories:

ROUTING, SECURITY, QOS, CONNECTIVITY, MONITORING, GENERAL

Rules:
- Choose the SINGLE most relevant category based on the question PRIMARY intent.
- If the question spans two domains, pick the one the question is MAINLY asking about.
- Use GENERAL only when the question genuinely does not fit any specific technical domain.
- After your reasoning, respond with a JSON block. No extra text after the JSON.

Response format:
```json
{\"category\": \"CATEGORY_NAME\", \"reason\": \"one short sentence\"}
```"""

print("Setup OK")

## Funciones

In [ ]:
def parse_json_from_response(text: str) -> dict | None:
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"\{[^{}]*\"category\"[^{}]*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


def classify(question: str) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question},
            ],
            temperature=0.3,
            max_tokens=256,
        )
        raw = response.choices[0].message.content
        parsed = parse_json_from_response(raw)
        if parsed and parsed.get("category") in CAT_LIST:
            return parsed["category"]
        for cat in CAT_LIST:
            if cat in raw.upper():
                return cat
        print(f"[WARN] No se pudo parsear: {repr(raw[:200])}")
        return "UNKNOWN"
    except Exception as e:
        print(f"[ERROR] {e}")
        return "UNKNOWN"

print("Funciones OK")

## Cargar dataset

In [ ]:
df = pd.read_csv(INPUT_CSV)
if SAMPLE:
    df = df.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
    print(f"Modo prueba: {SAMPLE} registros")

print(f"Total: {len(df)} preguntas")
df.head()

## Clasificar

In [ ]:
checkpoint = Path(CHECKPOINT)
results: dict[int, str] = {}
if checkpoint.exists():
    with open(checkpoint) as f:
        results = {int(k): v for k, v in json.load(f).items()}
    print(f"Checkpoint: {len(results)} registros ya clasificados.")

total = len(df)
pending = [i for i in range(total) if i not in results]
print(f"Pendientes: {len(pending)}")

for count, idx in enumerate(tqdm(pending)):
    question = str(df.loc[idx, "question"])
    results[idx] = classify(question)

    if (count + 1) % 50 == 0:
        with open(checkpoint, "w") as f:
            json.dump(results, f)

with open(checkpoint, "w") as f:
    json.dump(results, f)

print("Clasificación completa")

## Exportar y revisar resultados

In [ ]:
df["category"] = [results[i] for i in range(total)]
df.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset guardado: {OUTPUT_CSV}")

print("\n=== DISTRIBUCIÓN FINAL ===")
display(df["category"].value_counts().to_frame())

df.head(10)